# 1. Baseline : NLLB-200 (zero-shot) FR <-> Ewe

**Objectif** : mesurer la qualite de traduction du modele **NLLB-200-distilled-600M**
(Meta AI) sur notre corpus de test **sans aucun entrainement** (mode "zero-shot").

C'est la **reference de depart** : tout le travail de fine-tuning (notebook 2)
devra faire mieux que ces scores.

## Comment ca marche ?

- **NLLB** ("No Language Left Behind") est un modele de traduction multilingue
  entraine sur 200 langues, dont l'**ewe** (code `ewe_Latn`).
- Il est **"zero-shot"** pour nous : il n'a jamais vu notre corpus, mais il a vu
  de l'ewe pendant son entrainement.
- On mesure la qualite avec deux metriques standard :
  - **chrF++** (la metrique principale du projet, robuste aux petites variations)
  - **BLEU** (metrique classique, plus stricte)

> Le test set est charge depuis le **repo GitHub public** du projet.
> C'est le split `test.tsv` : 6 564 paires jamais utilisees pour l'entrainement.

In [ ]:
# Installation des bibliotheques necessaires
# - transformers : modeles HuggingFace (NLLB)
# - sacrebleu    : metriques chrF++ et BLEU
# - pandas       : lecture des fichiers TSV
# - sentencepiece : tokenizer de NLLB (obligatoire)
!pip install -q transformers sacrebleu pandas sentencepiece datasets

print("Dependances installees")

In [ ]:
# Diagnostic GPU
# Colab fournit un GPU (T4) gratuitement, mais il faut l'activer :
#   menu Executer > Changer le type d'execution > T4 GPU
#   puis Executer > Redemarrer la session (obligatoire).
import torch

print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("Memoire :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "Go")
else:
    print("Attention : execution sur CPU (lent). Active le GPU T4 puis redemarre la session.")
    print("Si Colab ne propose pas de GPU (quota), utilise Kaggle : Accelerator > GPU T4.")

In [ ]:
# Imports
import torch
import pandas as pd
import sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device utilise :", device)

In [ ]:
# Chargement du jeu de test depuis le repo GitHub public
URL_TEST = "https://raw.githubusercontent.com/cherif-tg/tg_nlp_toolkit/main/data/processed/v0.3/test.tsv"

try:
    df = pd.read_csv(URL_TEST, sep="\t")
    print("Test set charge :", len(df), "paires FR<->Ewe")
    print(df.head(3))
except Exception as e:
    print("Telechargement GitHub impossible :", e)
    print("Solution : telecharge test.tsv depuis le repo et execute :")
    print("  from google.colab import files; upload = files.upload()")

In [ ]:
# Chargement du modele NLLB-200-distilled-600M
# 600M parametres = version "distilled" (legere), adaptee a un GPU gratuit.
MODEL_NAME = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

# Verification des codes de langue
assert "fra_Latn" in tokenizer.additional_special_tokens, "francais absent"
assert "ewe_Latn" in tokenizer.additional_special_tokens, "ewe absent"
print("Modele charge - codes langue : fra_Latn (fr), ewe_Latn (ewe)")

In [ ]:
# Fonction de traduction en batch
# - src / tgt : codes de langue NLLB (fra_Latn, ewe_Latn)
# - num_beams=4 : recherche en faisceau (meilleure qualite que greedy)
# - Le tokenizer doit connaitre la langue SOURCE avant d'encoder.

def traduire(textes, src="fra_Latn", tgt="ewe_Latn", max_len=128, batch_size=16):
    tokenizer.src_lang = src
    resultats = []
    for i in range(0, len(textes), batch_size):
        lot = textes[i:i + batch_size]
        enc = tokenizer(lot, return_tensors="pt", padding=True,
                        truncation=True, max_length=max_len).to(device)
        with torch.no_grad():
            gen = model.generate(
                **enc,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt),
                max_new_tokens=max_len,
                num_beams=4,
            )
        resultats += tokenizer.batch_decode(gen, skip_special_tokens=True)
    return resultats

print("Fonction de traduction prete")

In [ ]:
# Evaluation FR -> EWE (le sens qui nous interesse le plus)
# On traduit les 6 564 phrases francaises du test set, puis on compare
# aux traductions ewe de reference avec chrF++ et BLEU.

preds_fr_ee = traduire(df["fr"].tolist(), src="fra_Latn", tgt="ewe_Latn")
refs_ee = df["ewe"].tolist()

chrf_fr_ee = sacrebleu.corpus.chrf(preds_fr_ee, [refs_ee])
bleu_fr_ee = sacrebleu.corpus.bleu(preds_fr_ee, [refs_ee])

print("FR -> EWE (zero-shot)")
print("   chrF++ :", round(chrf_fr_ee.score, 2))
print("   BLEU   :", round(bleu_fr_ee.score, 2))

# Afficher 3 exemples concrets
for i in range(3):
    print("--- Exemple", i + 1, "---")
    print("FR :", df['fr'].iloc[i])
    print("Ref:", refs_ee[i])
    print("Pred:", preds_fr_ee[i])

In [ ]:
# Evaluation EWE -> FR (sens inverse)
preds_ee_fr = traduire(df["ewe"].tolist(), src="ewe_Latn", tgt="fra_Latn")
refs_fr = df["fr"].tolist()

chrf_ee_fr = sacrebleu.corpus.chrf(preds_ee_fr, [refs_fr])
bleu_ee_fr = sacrebleu.corpus.bleu(preds_ee_fr, [refs_fr])

print("EWE -> FR (zero-shot)")
print("   chrF++ :", round(chrf_ee_fr.score, 2))
print("   BLEU   :", round(bleu_ee_fr.score, 2))

print("Tableau de bord baseline :")
print("   FR->EWE : chrF++", round(chrf_fr_ee.score, 2), "| BLEU", round(bleu_fr_ee.score, 2))
print("   EWE->FR : chrF++", round(chrf_ee_fr.score, 2), "| BLEU", round(bleu_ee_fr.score, 2))

## Comment interpreter ces scores ?

- **chrF++ 40-55** sur cette tache : le modele "se debrouille" (le vocabulaire
  religieux est bien connu de NLLB).
- **BLEU bas (< 15)** : normal, BLEU est tres strict sur les mots exacts, et
  l'ewe de 1913 a une orthographe differente de l'ewe moderne vu par NLLB.
- Ces scores sont notre **reference** : le notebook 2 (fine-tuning LoRA sur
  notre corpus) doit les **depasser**, surtout en chrF++.

> Si le score est tres bas, verifie que le GPU est actif
> (menu Executer > Changer le type d'execution > T4 GPU).